# Analyse de toutes les expériences exportées

Ce notebook recharge chaque expérience contenant `backtest_metrics.csv` dans `exports`, trace ses comparaisons par période et imprime uniquement les metrics clés.

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\dev\factor_backtest")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import func
importlib.reload(func)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    plot_performance_comparison,
    prepare_performance_comparisons_by_period,
)

print("Plugin chargé")

In [ ]:
EXPORT_ROOT = PLUGIN_DIR / "exports"
COMPARISON_MAX_TESTS = 8  # Augmentez cette valeur seulement si chaque figure reste lisible.
PERIOD_BREAKPOINTS = list(RECOMMENDED_PERIOD_BREAKPOINTS)
SHOW_PLOTS = True


In [ ]:
key_metric_columns = [
    "display_label", "test_path", "test_name", "test_type", "metric",
    "benchmark", "raw_variables", "composition_recipe",
    "actual_start_date", "actual_end_date", "observation_count", "years",
    "robust_score", "robust_score_rank_global", "robust_score_rank_within_type",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_sortino_ratio", "top_beta",
    "top_tracking_error", "top_information_ratio",
    "worst_annualized_return", "bench_annualized_return",
    "top_bench_ratio", "top_worst_ratio", "active_max_drawdown",
    "tracking_error_annualized", "min_rolling_3y_cagr",
    "active_cagr", "top_worst_cagr",
]

experiment_dirs = sorted(
    experiment_dir
    for experiment_dir in EXPORT_ROOT.iterdir()
    if experiment_dir.is_dir()
    and (experiment_dir / "backtest_metrics.csv").exists()
)
if not experiment_dirs:
    raise FileNotFoundError(
        f"Aucun dossier d'expérience avec backtest_metrics.csv dans {EXPORT_ROOT}"
    )

comparisons_by_period_by_experiment = {}
comparison_figures = {}

print("\n" + "=" * 100)
print(f"EXPORT_ROOT : {EXPORT_ROOT}")
print(f"Nombre d'expériences : {len(experiment_dirs)}")
print("Metrics clés de toutes les expériences, prêtes à copier dans un prompt")
print("=" * 100)

for experiment_dir in experiment_dirs:
    experiment_name = experiment_dir.name
    print("\n" + "@" * 100)
    print(f"EXPÉRIENCE : {experiment_name}")
    print(f"EXPORT_DIR : {experiment_dir}")

    comparisons_by_period = prepare_performance_comparisons_by_period(
        export_dir=experiment_dir,
        max_tests=COMPARISON_MAX_TESTS,
        period_breakpoints=PERIOD_BREAKPOINTS,
    )
    comparisons_by_period_by_experiment[experiment_name] = comparisons_by_period

    experiment_figures = {}
    for period_id, comparison in comparisons_by_period.items():
        comparison_figure = plot_performance_comparison(
            performance=comparison["performance"],
            ratios=comparison["ratios"],
            benchmark_column="Benchmark",
            title=f"{experiment_name} | Comparaison des performances",
            save_path=None,
            show_plot=SHOW_PLOTS,
            rebase=True,
            show_worst_performance=False,
            period_definitions=comparison["period_definitions"],
            default_period_id=period_id,
        )
        experiment_figures[period_id] = comparison_figure
    comparison_figures[experiment_name] = experiment_figures

    prompt_metrics = pd.read_csv(experiment_dir / "backtest_metrics.csv")
    for period_id, comparison in comparisons_by_period.items():
        period = comparison["period"]
        selected_top = [
            (label, test_path)
            for label, (test_path, portfolio)
            in comparison["performance_selection"].items()
            if portfolio == "Top"
        ]
        print("\n" + "#" * 100)
        print(f"PERIOD_ID : {period_id}")
        print(f"Période : {period['label']}")
        print(f"Début réel : {period.get('start')}")
        print(f"Fin réelle : {period.get('end')}")
        if not selected_top:
            print("Aucune performance Top sélectionnée.")
            continue

        selected_paths = [test_path for _, test_path in selected_top]
        display_labels = {label_path: label for label, label_path in selected_top}
        period_metrics = prompt_metrics.loc[
            prompt_metrics["period_id"].astype(str).eq(str(period_id))
            & prompt_metrics["test_path"].isin(selected_paths)
        ].copy()
        if not period_metrics.empty:
            period_metrics["_selection_order"] = period_metrics["test_path"].map(
                {test_path: index for index, test_path in enumerate(selected_paths)}
            )
            period_metrics["display_label"] = period_metrics["test_path"].map(
                display_labels
            )
            period_metrics = period_metrics.sort_values("_selection_order").drop(
                columns="_selection_order",
            )
        available_metric_columns = [
            column for column in key_metric_columns if column in period_metrics.columns
        ]
        print("Facteurs Top sélectionnés et metrics clés (CSV) :")
        print(period_metrics[available_metric_columns].to_csv(index=False))